In [5]:
"""
Traffic Demand Prediction — CatBoost-Dominant Ensemble
Metric : score = max(0, 100 * R2(actual, predicted))

Strategy:
  4 × CatBoost variants  (dominant — best single model)
    cat_deep    : depth=8,  lr=0.02,  l2=3,  rsm=1.0
    cat_shallow : depth=6,  lr=0.03,  l2=5,  rsm=0.8
    cat_wide    : depth=10, lr=0.015, l2=1,  rsm=0.9
    cat_reg     : depth=7,  lr=0.025, l2=10, rsm=0.85
  1 × LightGBM  (diversity)
  1 × XGBoost   (diversity)
  Ridge meta-learner on OOF predictions (positive=True)
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────
# 1.  LOAD
# ─────────────────────────────────────────────────────
train = pd.read_csv("/Users/nidhishgupta/Desktop/GridlockChallenge/data/raw/train.csv")
test = pd.read_csv("/Users/nidhishgupta/Desktop/GridlockChallenge/data/raw/test.csv")

TARGET = "demand"
ID_COL = "Index"

print("Train:", train.shape, "  Test:", test.shape)
print("\nTrain nulls:\n", train.isnull().sum())
print("\nTest  nulls:\n", test.isnull().sum())

# ─────────────────────────────────────────────────────
# 2.  FEATURE ENGINEERING  (light / moderate)
# ─────────────────────────────────────────────────────
CAT_COLS = ["geohash", "RoadType", "LargeVehicles", "Landmarks", "Weather"]


def engineer(df, is_train=True, geo_demand_map=None):
    df = df.copy()

    # ── Timestamp  '0:15' → hour, minute, cyclical
    ts    = df["timestamp"].astype(str).str.strip()
    parts = ts.str.split(":", expand=True)
    hour  = parts[0].astype(int)
    minute = parts[1].astype(int)
    mins  = hour * 60 + minute          # 0 … 1425
    total = 24 * 60

    df["hour"]         = hour
    df["minute"]       = minute
    df["time_minutes"] = mins
    df["time_sin"]     = np.sin(2 * np.pi * mins / total)
    df["time_cos"]     = np.cos(2 * np.pi * mins / total)

    # ── Day cyclic
    if df["day"].dtype == object:
        day_map = {"Monday": 0, "Tuesday": 1, "Wednesday": 2, "Thursday": 3,
                   "Friday": 4, "Saturday": 5, "Sunday": 6}
        df["day_num"] = df["day"].map(day_map).fillna(
            pd.to_numeric(df["day"], errors="coerce") % 7)
    else:
        df["day_num"] = df["day"].astype(float) % 7

    df["day_sin"]    = np.sin(2 * np.pi * df["day_num"] / 7)
    df["day_cos"]    = np.cos(2 * np.pi * df["day_num"] / 7)
    df["is_weekend"] = (df["day_num"] >= 5).astype(int)

    # ── Rush hour flags
    df["is_morning_rush"] = df["hour"].between(7, 10).astype(int)
    df["is_evening_rush"] = df["hour"].between(17, 20).astype(int)

    # ── Geohash × hour interaction  (captures location-specific peak patterns)
    df["geo_hour"] = df["geohash"].astype(str) + "_" + df["hour"].astype(str)

    # ── Temperature: fill with per-geohash median, fallback global median
    temp_geo    = df.groupby("geohash")["Temperature"].transform("median")
    temp_global = df["Temperature"].median()
    df["Temperature"] = df["Temperature"].fillna(temp_geo).fillna(temp_global)

    # ── NumberofLanes: fill with per-RoadType mode, fallback global mode
    def _mode(x):
        m = x.mode()
        return m[0] if len(m) else np.nan
    lane_road   = df.groupby("RoadType")["NumberofLanes"].transform(_mode)
    lane_global = df["NumberofLanes"].mode()[0]
    df["NumberofLanes"] = df["NumberofLanes"].fillna(lane_road).fillna(lane_global)

    # ── Geohash mean-demand target encoding  (leak-safe: only from train)
    if is_train:
        geo_demand_map = df.groupby("geohash")[TARGET].mean().to_dict()
    global_mean = np.mean(list(geo_demand_map.values()))
    df["geo_mean_demand"] = df["geohash"].map(geo_demand_map).fillna(global_mean)

    # ── Categorical null fill
    for c in CAT_COLS:
        df[c] = df[c].fillna("Unknown").astype(str)

    return df, geo_demand_map


train, geo_map = engineer(train, is_train=True)
test,  _       = engineer(test,  is_train=False, geo_demand_map=geo_map)

# ─────────────────────────────────────────────────────
# 3.  FEATURE LIST & TWO VERSIONS OF X
#     CatBoost  → raw string categoricals in a DataFrame (cb.Pool handles them)
#     LGB/XGB   → label-encoded integers
# ─────────────────────────────────────────────────────
DROP  = [ID_COL, TARGET, "timestamp", "day"]
FEATS = [c for c in train.columns if c not in DROP]

CAT_STR_COLS = [c for c in (CAT_COLS + ["geo_hour"]) if c in FEATS]
CAT_IDX      = [FEATS.index(c) for c in CAT_STR_COLS]   # kept for reference

print("\nFeatures used:", FEATS)
print("Categorical cols for CatBoost:", CAT_STR_COLS)

# ── CatBoost version: keep strings, fill remaining numeric NaNs
train_cat = train[FEATS].copy()
test_cat  = test[FEATS].copy()
for c in CAT_STR_COLS:
    train_cat[c] = train_cat[c].astype(str)
    test_cat[c]  = test_cat[c].astype(str)

# ── LGB / XGB version: label-encode categoricals → integers
train_enc = train[FEATS].copy()
test_enc  = test[FEATS].copy()
le_dict   = {}
for col in CAT_STR_COLS:
    le = LabelEncoder()
    combined = pd.concat([train_enc[col], test_enc[col]], axis=0).astype(str)
    le.fit(combined)
    train_enc[col] = le.transform(train_enc[col].astype(str))
    test_enc[col]  = le.transform(test_enc[col].astype(str))
    le_dict[col]   = le

y      = train[TARGET].values
X_test_enc = test_enc[FEATS].values
X_enc      = train_enc[FEATS].values

# ─────────────────────────────────────────────────────
# 4.  MODEL CONFIG
# ─────────────────────────────────────────────────────
# Each CatBoost variant differs in depth, lr, l2, rsm (column sampling), seed
# → low correlation between their errors → stacking gains are real
CAT_CONFIGS = {
    "cat_deep":    dict(depth=8,  lr=0.02,  iters=2500, l2=3,  rsm=1.0,  seed=42),
    "cat_shallow": dict(depth=6,  lr=0.03,  iters=2500, l2=5,  rsm=0.8,  seed=7),
    "cat_wide":    dict(depth=10, lr=0.015, iters=3000, l2=1,  rsm=0.9,  seed=21),
    "cat_reg":     dict(depth=7,  lr=0.025, iters=2500, l2=10, rsm=0.85, seed=99),
}


def make_catboost(cfg):
    # cat_features are passed via cb.Pool, NOT in the constructor
    return cb.CatBoostRegressor(
        iterations=cfg["iters"],
        learning_rate=cfg["lr"],
        depth=cfg["depth"],
        l2_leaf_reg=cfg["l2"],
        rsm=cfg["rsm"],
        border_count=128,
        eval_metric="R2",
        random_seed=cfg["seed"],
        verbose=0,
        od_type="Iter",
        od_wait=50,
    )


def make_lgb():
    return lgb.LGBMRegressor(
        n_estimators=2000, learning_rate=0.02, num_leaves=127,
        subsample=0.8, colsample_bytree=0.8,
        min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=-1,
    )


def make_xgb():
    return xgb.XGBRegressor(
        n_estimators=2000, learning_rate=0.02, max_depth=7,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=3, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, tree_method="hist", verbosity=0,
    )


ALL_MODEL_NAMES = list(CAT_CONFIGS.keys()) + ["lgb", "xgb"]

# ─────────────────────────────────────────────────────
# 5.  5-FOLD OOF STACKING
# ─────────────────────────────────────────────────────
N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds  = {n: np.zeros(len(X_enc))      for n in ALL_MODEL_NAMES}
test_preds = {n: np.zeros(len(X_test_enc)) for n in ALL_MODEL_NAMES}

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_enc, y)):
    print(f"\n══ Fold {fold + 1}/{N_FOLDS} ══")

    # ── encoded arrays for LGB / XGB
    X_tr_enc,  X_val_enc = X_enc[tr_idx],  X_enc[val_idx]
    y_tr,      y_val     = y[tr_idx],      y[val_idx]

    # ── string DataFrames for CatBoost pools
    X_tr_cat  = train_cat.iloc[tr_idx][FEATS]
    X_val_cat = train_cat.iloc[val_idx][FEATS]

    pool_tr  = cb.Pool(X_tr_cat,  y_tr,  cat_features=CAT_STR_COLS)
    pool_val = cb.Pool(X_val_cat, y_val, cat_features=CAT_STR_COLS)

    for name in ALL_MODEL_NAMES:
        if name in CAT_CONFIGS:
            m = make_catboost(CAT_CONFIGS[name])
            m.fit(pool_tr, eval_set=pool_val, verbose=0)
            val_pred  = m.predict(X_val_cat)
            test_pred = m.predict(
                cb.Pool(test_cat[FEATS], cat_features=CAT_STR_COLS)
            )

        elif name == "lgb":
            m = make_lgb()
            m.fit(X_tr_enc, y_tr,
                  eval_set=[(X_val_enc, y_val)],
                  callbacks=[lgb.early_stopping(50, verbose=False),
                              lgb.log_evaluation(-1)])
            val_pred  = m.predict(X_val_enc)
            test_pred = m.predict(X_test_enc)

        else:  # xgb
            m = make_xgb()
            m.fit(X_tr_enc, y_tr,
                  eval_set=[(X_val_enc, y_val)],
                  verbose=False)
            val_pred  = m.predict(X_val_enc)
            test_pred = m.predict(X_test_enc)

        oof_preds[name][val_idx]  = val_pred
        test_preds[name]         += test_pred / N_FOLDS

        r2 = r2_score(y_val, val_pred)
        print(f"  {name:<14}  fold-R²={r2:.5f}  score={max(0, 100*r2):.2f}")

# ─────────────────────────────────────────────────────
# 6.  OOF SUMMARY
# ─────────────────────────────────────────────────────
print("\n══ Full OOF R² ══")
for name in ALL_MODEL_NAMES:
    r2 = r2_score(y, oof_preds[name])
    print(f"  {name:<14}  OOF R²={r2:.5f}  score={max(0, 100*r2):.2f}")

# ─────────────────────────────────────────────────────
# 7.  META-LEARNER  (Ridge, positive weights enforced)
# ─────────────────────────────────────────────────────
oof_mat  = np.column_stack([oof_preds[n]  for n in ALL_MODEL_NAMES])
test_mat = np.column_stack([test_preds[n] for n in ALL_MODEL_NAMES])

meta = Ridge(alpha=0.5, fit_intercept=True, positive=True)
meta.fit(oof_mat, y)
stacked_oof  = meta.predict(oof_mat)
stacked_test = meta.predict(test_mat)
stack_r2 = r2_score(y, stacked_oof)
print(f"\n  Ridge-stacked OOF R²={stack_r2:.5f}  score={max(0, 100*stack_r2):.2f}")
print("  Meta weights:", {n: round(w, 4)
                           for n, w in zip(ALL_MODEL_NAMES, meta.coef_)})

# ── Weighted average fallback (weight ∝ OOF R²)
oof_r2s  = np.array([max(0, r2_score(y, oof_preds[n])) for n in ALL_MODEL_NAMES])
weights  = oof_r2s / oof_r2s.sum()
avg_oof  = oof_mat  @ weights
avg_test = test_mat @ weights
avg_r2   = r2_score(y, avg_oof)
print(f"  Weighted-avg OOF R²={avg_r2:.5f}  score={max(0, 100*avg_r2):.2f}")
print("  Avg weights:", {n: round(w, 4)
                          for n, w in zip(ALL_MODEL_NAMES, weights)})

# ── Best strategy
if stack_r2 >= avg_r2:
    final_preds = stacked_test
    chosen = "Ridge stacking"
else:
    final_preds = avg_test
    chosen = "Weighted average"
print(f"\n→ Using: {chosen}")

# ─────────────────────────────────────────────────────
# 8.  SUBMISSION
# ─────────────────────────────────────────────────────
sub = pd.DataFrame({ID_COL: test[ID_COL], TARGET: final_preds})
sub.to_csv("submission.csv", index=False)
print(f"\nSaved submission.csv  shape={sub.shape}")
print(sub.head())

Train: (77299, 11)   Test: (41778, 10)

Train nulls:
 Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

Test  nulls:
 Index               0
geohash             0
day                 0
timestamp           0
RoadType          324
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      1349
Weather           431
dtype: int64

Features used: ['geohash', 'RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks', 'Temperature', 'Weather', 'hour', 'minute', 'time_minutes', 'time_sin', 'time_cos', 'day_num', 'day_sin', 'day_cos', 'is_weekend', 'is_morning_rush', 'is_evening_rush', 'geo_hour', 'geo_mean_demand']
Categorical cols for CatBoost: ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geo_hour']

══ Fold 1/5 ══
  cat_deep        fold-R²=0.9484